# 1.1 Data Analysis

In [1]:
from pathlib import Path
from collections import OrderedDict

# Try common locations for the raw image dataset.
possible_dirs = [
    Path.cwd() / "data" / "1_raw",          # if notebook runs from project root
    Path.cwd().parent / "data" / "1_raw",   # if notebook runs from processing/
]

raw_dir = next((p for p in possible_dirs if p.exists()), None)
if raw_dir is None:
    raise FileNotFoundError(
        "Could not find data/1_raw. Please provide the dataset."
    )

image_exts = {".jpg", ".jpeg", ".png", ".heic"}
counts = OrderedDict()

for folder in sorted([p for p in raw_dir.iterdir() if p.is_dir()]):
    num_images = sum(1 for f in folder.iterdir() if f.is_file() and f.suffix.lower() in image_exts)
    counts[folder.name] = num_images

target = 30

print(f"Dataset folder: {raw_dir}")
print("=" * 40)
for name, count in counts.items():
    remaining = max(0, target - count)
    status = "done" if remaining == 0 else f"need {remaining} more"
    print(f"{name:<15} : {count:>3} images | {status}")

print("=" * 40)
print(f"Total folders: {len(counts)}")
print(f"Total images : {sum(counts.values())}")


Dataset folder: c:\Users\harry\Documents\school\ip\AttSystem\data\1_raw
benjamin        :  30 images | done
chern_tak       :  28 images | need 2 more
chillien        :   9 images | need 21 more
daniel          :  30 images | done
dylan           :  21 images | need 9 more
han_soon        :  20 images | need 10 more
harry           :  10 images | need 20 more
isaac           :  10 images | need 20 more
jing_ang        :  30 images | done
jun_wei         :  27 images | need 3 more
kang_kai        :  30 images | done
marion          :  28 images | need 2 more
ms_nurul        :  11 images | need 19 more
qi_xuan         :  30 images | done
shuang_quan     :  45 images | done
wee_xuan        :  24 images | need 6 more
xiang_yue       :  30 images | done
xu_sheng        :  29 images | need 1 more
yoke_hong       :  32 images | done
yong_kang       :  24 images | need 6 more
zi_herng        :  10 images | need 20 more
Total folders: 21
Total images : 508


# 1.2 Image Preprocessing

### 1.2.1 Conversion and Standardization

In [2]:
from pathlib import Path
from PIL import Image, UnidentifiedImageError
from pillow_heif import register_heif_opener

register_heif_opener()

source_dir = Path.cwd() / "data" / "1_raw"
if not source_dir.exists():
    source_dir = Path.cwd().parent / "data" / "1_raw"

standardized_dir = Path.cwd() / "data" / "2_standarized"
if not standardized_dir.exists():
    standardized_dir = Path.cwd().parent / "data" / "2_standarized"

if not source_dir.exists():
    print("Missing data/1_raw. Please run section 1 first.")
else:
    raw_image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".webp", ".heic", ".heif"}
    raw_image_files = [
        file_path
        for class_dir in sorted([p for p in source_dir.iterdir() if p.is_dir()])
        for file_path in sorted([p for p in class_dir.iterdir() if p.is_file() and p.suffix.lower() in raw_image_exts])
    ]

    if not raw_image_files:
        print("No raw image files were found in data/1_raw. Please run section 1 first.")
    else:
        standardized_dir.mkdir(parents=True, exist_ok=True)

        for class_dir in sorted([p for p in source_dir.iterdir() if p.is_dir()]):
            input_files = sorted([p for p in class_dir.iterdir() if p.is_file() and p.suffix.lower() in raw_image_exts])
            if not input_files:
                continue

            output_class_dir = standardized_dir / class_dir.name
            output_class_dir.mkdir(parents=True, exist_ok=True)

            for index, file_path in enumerate(input_files, start=1):
                output_path = output_class_dir / f"{class_dir.name}_{index:03d}.jpg"

                try:
                    with Image.open(file_path) as image:
                        rgb_image = image.convert("RGB")
                        rgb_image.save(output_path, format="JPEG", quality=95)
                except UnidentifiedImageError:
                    print(f"Skipped non-image file: {file_path}")
                except Exception as error:
                    print(f"Failed to process {file_path}: {error}")

        print(f"Standardized images saved to: {standardized_dir}")
        print("All valid files were converted to standardized RGB JPG format.")

Standardized images saved to: c:\Users\harry\Documents\school\ip\AttSystem\data\2_standarized
All valid files were converted to standardized RGB JPG format.


### 1.2.2 Image Splitting (Before ROI)

In [3]:
from pathlib import Path
import random
import shutil

standardized_dir = Path.cwd() / "data" / "2_standarized"
if not standardized_dir.exists():
    standardized_dir = Path.cwd().parent / "data" / "2_standarized"

train_dir = Path.cwd() / "data" / "3_train"
test_dir = Path.cwd() / "data" / "3_test"

if not train_dir.parent.exists():
    train_dir = Path.cwd().parent / "data" / "3_train"
    test_dir = Path.cwd().parent / "data" / "3_test"

if not standardized_dir.exists():
    print("Missing data/2_standarized. Please run section 2.1 first.")
else:
    standardized_classes = [p for p in standardized_dir.iterdir() if p.is_dir()]
    standardized_images = [file_path for class_dir in standardized_classes for file_path in class_dir.iterdir() if file_path.is_file() and file_path.suffix.lower() == ".jpg"]

    if not standardized_images:
        print("No standardized JPG images were found in data/2_standarized. Please run section 2.1 first.")
    else:
        train_dir.mkdir(parents=True, exist_ok=True)
        test_dir.mkdir(parents=True, exist_ok=True)

        for class_dir in sorted(standardized_classes):
            images = sorted([p for p in class_dir.iterdir() if p.is_file() and p.suffix.lower() == ".jpg"])

            if not images:
                continue

            random.seed(42)
            random.shuffle(images)
            split_index = int(len(images) * 0.8)
            train_files = images[:split_index]
            test_files = images[split_index:]

            train_class_dir = train_dir / class_dir.name
            test_class_dir = test_dir / class_dir.name
            train_class_dir.mkdir(parents=True, exist_ok=True)
            test_class_dir.mkdir(parents=True, exist_ok=True)

            for index, file_path in enumerate(train_files, start=1):
                output_name = f"{class_dir.name}_train_{index:03d}.jpg"
                shutil.copy2(file_path, train_class_dir / output_name)

            for index, file_path in enumerate(test_files, start=1):
                output_name = f"{class_dir.name}_test_{index:03d}.jpg"
                shutil.copy2(file_path, test_class_dir / output_name)

        print(f"Train images saved to: {train_dir}")
        print(f"Test images saved to: {test_dir}")

Train images saved to: c:\Users\harry\Documents\school\ip\AttSystem\data\3_train
Test images saved to: c:\Users\harry\Documents\school\ip\AttSystem\data\3_test


# 1.3 ROI Preprocessing (Split-Aware)

In [4]:
from pathlib import Path
import cv2

train_input_dir = Path.cwd() / "data" / "3_train"
test_input_dir = Path.cwd() / "data" / "3_test"
if not train_input_dir.exists() or not test_input_dir.exists():
    train_input_dir = Path.cwd().parent / "data" / "3_train"
    test_input_dir = Path.cwd().parent / "data" / "3_test"

processed_train_dir = Path.cwd() / "data" / "4_processed_train"
processed_test_dir = Path.cwd() / "data" / "4_processed_test"
if not processed_train_dir.parent.exists():
    processed_train_dir = Path.cwd().parent / "data" / "4_processed_train"
    processed_test_dir = Path.cwd().parent / "data" / "4_processed_test"

roi_ready_train_dir = Path.cwd() / "data" / "4_roi_ready_train"
roi_ready_test_dir = Path.cwd() / "data" / "4_roi_ready_test"
if not roi_ready_train_dir.parent.exists():
    roi_ready_train_dir = Path.cwd().parent / "data" / "4_roi_ready_train"
    roi_ready_test_dir = Path.cwd().parent / "data" / "4_roi_ready_test"

if not train_input_dir.exists() or not test_input_dir.exists():
    print("Missing split folders data/3_train or data/3_test. Please run section 2.2 first.")
else:
    split_configs = [
        ("train", train_input_dir, processed_train_dir, roi_ready_train_dir),
        ("test", test_input_dir, processed_test_dir, roi_ready_test_dir),
    ]

    for split_name, input_root, processed_root, roi_ready_root in split_configs:
        split_classes = [p for p in input_root.iterdir() if p.is_dir()]
        split_images = [
            file_path
            for class_dir in split_classes
            for file_path in class_dir.iterdir()
            if file_path.is_file() and file_path.suffix.lower() == ".jpg"
        ]

        if not split_images:
            print(f"No JPG images found in {input_root}. Please run section 2.2 first.")
            continue

        processed_root.mkdir(parents=True, exist_ok=True)
        roi_ready_root.mkdir(parents=True, exist_ok=True)

        for class_dir in sorted(split_classes):
            input_files = sorted([p for p in class_dir.iterdir() if p.is_file() and p.suffix.lower() == ".jpg"])
            if not input_files:
                continue

            processed_class_dir = processed_root / class_dir.name
            roi_ready_class_dir = roi_ready_root / class_dir.name
            processed_class_dir.mkdir(parents=True, exist_ok=True)
            roi_ready_class_dir.mkdir(parents=True, exist_ok=True)

            clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

            for index, file_path in enumerate(input_files, start=1):
                image = cv2.imread(str(file_path), cv2.IMREAD_COLOR)
                if image is None:
                    print(f"Skipped unreadable image: {file_path}")
                    continue

                # Step 1: base processed image (grayscale + CLAHE).
                gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
                enhanced = clahe.apply(gray)

                processed_path = processed_class_dir / f"{class_dir.name}_{split_name}_processed_{index:03d}.jpg"
                cv2.imwrite(str(processed_path), enhanced, [int(cv2.IMWRITE_JPEG_QUALITY), 95])

                # Step 2: ROI-ready image optimized for face detection.
                denoised = cv2.GaussianBlur(enhanced, (3, 3), 0)
                roi_ready = denoised

                min_face_dim_target = 180
                h, w = roi_ready.shape[:2]
                short_side = min(h, w)
                if short_side < min_face_dim_target:
                    scale = min_face_dim_target / max(1.0, float(short_side))
                    new_w = int(w * scale)
                    new_h = int(h * scale)
                    roi_ready = cv2.resize(roi_ready, (new_w, new_h), interpolation=cv2.INTER_CUBIC)

                roi_ready_path = roi_ready_class_dir / f"{class_dir.name}_{split_name}_roi_ready_{index:03d}.jpg"
                cv2.imwrite(str(roi_ready_path), roi_ready, [int(cv2.IMWRITE_JPEG_QUALITY), 95])

        print(f"{split_name.capitalize()} processed images saved to: {processed_root}")
        print(f"{split_name.capitalize()} ROI-ready images saved to: {roi_ready_root}")

    print("Split-aware preprocessing + ROI-ready preparation completed.")

Train processed images saved to: c:\Users\harry\Documents\school\ip\AttSystem\data\4_processed_train
Train ROI-ready images saved to: c:\Users\harry\Documents\school\ip\AttSystem\data\4_roi_ready_train
Test processed images saved to: c:\Users\harry\Documents\school\ip\AttSystem\data\4_processed_test
Test ROI-ready images saved to: c:\Users\harry\Documents\school\ip\AttSystem\data\4_roi_ready_test
Split-aware preprocessing + ROI-ready preparation completed.


## 1.4 Face ROI Extraction

In [5]:
from pathlib import Path
import cv2
import numpy as np

# Section 4 should only perform face ROI extraction from ROI-ready images.
roi_ready_train_dir = Path.cwd() / "data" / "4_roi_ready_train"
roi_ready_test_dir = Path.cwd() / "data" / "4_roi_ready_test"
if not roi_ready_train_dir.exists() or not roi_ready_test_dir.exists():
    roi_ready_train_dir = Path.cwd().parent / "data" / "4_roi_ready_train"
    roi_ready_test_dir = Path.cwd().parent / "data" / "4_roi_ready_test"

roi_train_dir = Path.cwd() / "data" / "5_roi_train"
roi_test_dir = Path.cwd() / "data" / "5_roi_test"
if not roi_train_dir.parent.exists():
    roi_train_dir = Path.cwd().parent / "data" / "5_roi_train"
    roi_test_dir = Path.cwd().parent / "data" / "5_roi_test"

if not roi_ready_train_dir.exists() or not roi_ready_test_dir.exists():
    print("Missing data/4_roi_ready_train or data/4_roi_ready_test. Please run section 3 first.")
else:
    face_cascade_names = [
        "haarcascade_frontalface_default.xml",
        "haarcascade_frontalface_alt2.xml",
    ]
    face_detectors = []
    for name in face_cascade_names:
        cascade_path = Path(cv2.data.haarcascades) / name
        detector = cv2.CascadeClassifier(str(cascade_path))
        if not detector.empty():
            face_detectors.append(detector)

    eye_cascade_path = Path(cv2.data.haarcascades) / "haarcascade_eye.xml"
    eye_detector = cv2.CascadeClassifier(str(eye_cascade_path))

    if not face_detectors:
        raise RuntimeError("Could not load OpenCV face cascades.")
    if eye_detector.empty():
        raise RuntimeError("Could not load OpenCV eye cascade.")

    def get_square_roi(gray_image: np.ndarray, x: int, y: int, w: int, h: int, expand: float = 1.75):
        img_h, img_w = gray_image.shape[:2]

        # Keep crop center close to face center so both upper and lower face stay included.
        cx = x + (w // 2)
        cy = y + int(h * 0.50)

        side = int(max(w, h) * expand)
        side = max(side, 128)
        half = side // 2

        # Pad image before crop to avoid truncation when the face is near borders.
        pad = max(64, int(side * 0.35))
        padded = cv2.copyMakeBorder(
            gray_image,
            pad,
            pad,
            pad,
            pad,
            borderType=cv2.BORDER_REFLECT_101,
        )

        cx_p = cx + pad
        cy_p = cy + pad

        x1 = max(0, cx_p - half)
        y1 = max(0, cy_p - half)
        x2 = x1 + side
        y2 = y1 + side

        if x2 > padded.shape[1]:
            x2 = padded.shape[1]
            x1 = max(0, x2 - side)
        if y2 > padded.shape[0]:
            y2 = padded.shape[0]
            y1 = max(0, y2 - side)

        roi = padded[y1:y2, x1:x2]
        if roi.size == 0:
            return roi

        # Standardize output size for more stable downstream training.
        return cv2.resize(roi, (220, 220), interpolation=cv2.INTER_CUBIC)

    def eye_evidence(gray_image: np.ndarray, x: int, y: int, w: int, h: int) -> int:
        face_roi = gray_image[y:y + h, x:x + w]
        if face_roi.size == 0:
            return 0

        upper = face_roi[:max(1, int(h * 0.62)), :]
        min_eye_w = max(10, int(w * 0.11))
        min_eye_h = max(8, int(h * 0.08))

        eyes = eye_detector.detectMultiScale(
            upper,
            scaleFactor=1.08,
            minNeighbors=5,
            minSize=(min_eye_w, min_eye_h),
        )

        return min(2, len(eyes))

    def score_candidate(x: int, y: int, w: int, h: int, img_w: int, img_h: int, eye_count: int, weight: float = 0.0) -> float:
        img_area = float(img_h * img_w)
        area_ratio = (w * h) / max(1.0, img_area)

        box_cx = x + (w / 2.0)
        box_cy = y + (h / 2.0)
        center_dx = abs(box_cx - (img_w / 2.0)) / max(1.0, img_w / 2.0)
        center_dy = abs(box_cy - (img_h / 2.0)) / max(1.0, img_h / 2.0)
        center_penalty = center_dx + (1.10 * center_dy)

        eye_bonus = 0.45 * float(eye_count)

        return (4.1 * area_ratio) + eye_bonus + (0.10 * float(weight)) - (0.78 * center_penalty)

    def pick_best_face(gray_image: np.ndarray):
        img_h, img_w = gray_image.shape[:2]
        equalized = cv2.equalizeHist(gray_image)
        variants = [gray_image, equalized]

        pass_configs = [
            {"scaleFactor": 1.05, "minNeighbors": 8, "size_ratio": 0.22, "min_area": 0.09, "aspect_min": 0.72, "aspect_max": 1.35, "require_eye": True},
            {"scaleFactor": 1.03, "minNeighbors": 7, "size_ratio": 0.18, "min_area": 0.06, "aspect_min": 0.62, "aspect_max": 1.50, "require_eye": True},
            {"scaleFactor": 1.03, "minNeighbors": 6, "size_ratio": 0.16, "min_area": 0.05, "aspect_min": 0.58, "aspect_max": 1.60, "require_eye": True},
        ]

        min_abs_w = int(img_w * 0.16)
        min_abs_h = int(img_h * 0.16)
        edge_margin_px = max(6, int(min(img_w, img_h) * 0.03))

        for cfg in pass_configs:
            candidates = []
            min_w = max(36, int(img_w * cfg["size_ratio"]), min_abs_w)
            min_h = max(36, int(img_h * cfg["size_ratio"]), min_abs_h)

            for detector in face_detectors:
                for image_variant in variants:
                    try:
                        faces, _, weights = detector.detectMultiScale3(
                            image_variant,
                            scaleFactor=cfg["scaleFactor"],
                            minNeighbors=cfg["minNeighbors"],
                            minSize=(min_w, min_h),
                            outputRejectLevels=True,
                        )
                        flat_weights = weights.flatten() if len(weights) else np.zeros(len(faces), dtype=np.float32)
                    except Exception:
                        faces = detector.detectMultiScale(
                            image_variant,
                            scaleFactor=cfg["scaleFactor"],
                            minNeighbors=cfg["minNeighbors"],
                            minSize=(min_w, min_h),
                        )
                        flat_weights = np.zeros(len(faces), dtype=np.float32)

                    for (x, y, w, h), weight in zip(faces, flat_weights):
                        x, y, w, h = int(x), int(y), int(w), int(h)

                        if w < min_abs_w or h < min_abs_h:
                            continue

                        area_ratio = (w * h) / max(1.0, float(img_h * img_w))
                        aspect = w / max(1.0, float(h))

                        if area_ratio < cfg["min_area"]:
                            continue
                        if aspect < cfg["aspect_min"] or aspect > cfg["aspect_max"]:
                            continue

                        center_y = y + (h / 2.0)
                        if center_y > (img_h * 0.74):
                            continue

                        # Reject detections that touch image borders; these are often partial faces.
                        if x <= edge_margin_px or y <= edge_margin_px:
                            continue
                        if (x + w) >= (img_w - edge_margin_px) or (y + h) >= (img_h - edge_margin_px):
                            continue

                        # Reject tiny top-strip detections that commonly lock onto eyebrow/eye region only.
                        if y < (img_h * 0.10) and h < (img_h * 0.24):
                            continue

                        eyes = eye_evidence(gray_image, x, y, w, h)
                        if cfg["require_eye"] and eyes == 0:
                            continue

                        score = score_candidate(x, y, w, h, img_w, img_h, eyes, float(weight))
                        candidates.append((score, (x, y, w, h)))

            if candidates:
                candidates.sort(key=lambda item: item[0], reverse=True)
                return candidates[0][1]

        return None

    split_configs = [
        ("train", roi_ready_train_dir, roi_train_dir),
        ("test", roi_ready_test_dir, roi_test_dir),
    ]

    for split_name, input_root, output_root in split_configs:
        output_root.mkdir(parents=True, exist_ok=True)

        total_images = 0
        saved_rois = 0
        no_face = 0
        unreadable = 0

        for class_dir in sorted([p for p in input_root.iterdir() if p.is_dir()]):
            input_files = sorted([p for p in class_dir.iterdir() if p.is_file() and p.suffix.lower() == ".jpg"])
            if not input_files:
                continue

            output_class_dir = output_root / class_dir.name
            output_class_dir.mkdir(parents=True, exist_ok=True)

            for old_file in output_class_dir.glob("*.jpg"):
                old_file.unlink(missing_ok=True)

            for index, file_path in enumerate(input_files, start=1):
                total_images += 1

                gray = cv2.imread(str(file_path), cv2.IMREAD_GRAYSCALE)
                if gray is None:
                    unreadable += 1
                    print(f"Skipped unreadable image: {file_path}")
                    continue

                best_face = pick_best_face(gray)
                if best_face is None:
                    no_face += 1
                    continue

                x, y, w, h = best_face
                roi = get_square_roi(gray, x, y, w, h, expand=1.75)

                if roi.size == 0:
                    no_face += 1
                    continue

                if min(roi.shape[:2]) < 128:
                    no_face += 1
                    continue

                output_path = output_class_dir / f"{class_dir.name}_{split_name}_roi_{index:03d}.jpg"
                cv2.imwrite(str(output_path), roi, [int(cv2.IMWRITE_JPEG_QUALITY), 95])
                saved_rois += 1

        print(f"[{split_name}] input images : {total_images}")
        print(f"[{split_name}] saved ROIs   : {saved_rois}")
        print(f"[{split_name}] no face      : {no_face}")
        print(f"[{split_name}] unreadable   : {unreadable}")
        print(f"[{split_name}] output folder: {output_root}")
        print("-" * 50)

    print("Face ROI extraction completed for both train and test splits.")

[train] input images : 402
[train] saved ROIs   : 299
[train] no face      : 103
[train] unreadable   : 0
[train] output folder: c:\Users\harry\Documents\school\ip\AttSystem\data\5_roi_train
--------------------------------------------------
[test] input images : 106
[test] saved ROIs   : 83
[test] no face      : 23
[test] unreadable   : 0
[test] output folder: c:\Users\harry\Documents\school\ip\AttSystem\data\5_roi_test
--------------------------------------------------
Face ROI extraction completed for both train and test splits.


# 1.5 Final ROI Enhancement

In [6]:
from pathlib import Path
import cv2
import numpy as np

# Final ROI enhancement for train/test images.
roi_train_dir = Path.cwd() / "data" / "5_roi_train"
roi_test_dir = Path.cwd() / "data" / "5_roi_test"
if not roi_train_dir.exists() or not roi_test_dir.exists():
    roi_train_dir = Path.cwd().parent / "data" / "5_roi_train"
    roi_test_dir = Path.cwd().parent / "data" / "5_roi_test"

enhanced_train_dir = Path.cwd() / "data" / "6_enhanced_roi_train"
enhanced_test_dir = Path.cwd() / "data" / "6_enhanced_roi_test"
if not enhanced_train_dir.parent.exists():
    enhanced_train_dir = Path.cwd().parent / "data" / "6_enhanced_roi_train"
    enhanced_test_dir = Path.cwd().parent / "data" / "6_enhanced_roi_test"

if not roi_train_dir.exists() or not roi_test_dir.exists():
    print("Missing data/5_roi_train or data/5_roi_test. Please run section 4 first.")
else:
    def enhance_roi(gray_image: np.ndarray) -> np.ndarray:
        clahe = cv2.createCLAHE(clipLimit=2.2, tileGridSize=(8, 8))
        local_contrast = clahe.apply(gray_image)
        denoised = cv2.bilateralFilter(local_contrast, 5, 35, 35)
        blurred = cv2.GaussianBlur(denoised, (0, 0), 1.0)
        sharpened = cv2.addWeighted(denoised, 1.25, blurred, -0.25, 0)
        normalized = cv2.normalize(sharpened, None, 0, 255, cv2.NORM_MINMAX)
        return normalized.astype(np.uint8)

    enhance_configs = [
        ("train", roi_train_dir, enhanced_train_dir),
        ("test", roi_test_dir, enhanced_test_dir),
    ]

    for split_name, input_root, output_root in enhance_configs:
        output_root.mkdir(parents=True, exist_ok=True)

        total_images = 0
        enhanced_images = 0
        unreadable = 0

        for class_dir in sorted([p for p in input_root.iterdir() if p.is_dir()]):
            input_files = sorted([p for p in class_dir.iterdir() if p.is_file() and p.suffix.lower() == ".jpg"])
            if not input_files:
                continue

            output_class_dir = output_root / class_dir.name
            output_class_dir.mkdir(parents=True, exist_ok=True)

            for old_file in output_class_dir.glob("*.jpg"):
                old_file.unlink(missing_ok=True)

            for index, file_path in enumerate(input_files, start=1):
                total_images += 1

                gray = cv2.imread(str(file_path), cv2.IMREAD_GRAYSCALE)
                if gray is None:
                    unreadable += 1
                    print(f"Skipped unreadable image: {file_path}")
                    continue

                enhanced = enhance_roi(gray)
                output_path = output_class_dir / f"{class_dir.name}_{split_name}_enhanced_{index:03d}.jpg"
                cv2.imwrite(str(output_path), enhanced, [int(cv2.IMWRITE_JPEG_QUALITY), 95])
                enhanced_images += 1

        print(f"[{split_name}] input images     : {total_images}")
        print(f"[{split_name}] enhanced images  : {enhanced_images}")
        print(f"[{split_name}] unreadable       : {unreadable}")
        print(f"[{split_name}] output folder    : {output_root}")
        print("-" * 50)

    print("Final ROI enhancement completed for both train and test splits.")

[train] input images     : 299
[train] enhanced images  : 299
[train] unreadable       : 0
[train] output folder    : c:\Users\harry\Documents\school\ip\AttSystem\data\6_enhanced_roi_train
--------------------------------------------------
[test] input images     : 83
[test] enhanced images  : 83
[test] unreadable       : 0
[test] output folder    : c:\Users\harry\Documents\school\ip\AttSystem\data\6_enhanced_roi_test
--------------------------------------------------
Final ROI enhancement completed for both train and test splits.


# 1.6 Dataset Analysis

In [7]:
from pathlib import Path
from collections import OrderedDict

# Dataset analysis by person across the full pipeline.
project_root = Path.cwd()
if not (project_root / "data").exists():
    project_root = Path.cwd().parent

data_root = project_root / "data"
if not data_root.exists():
    raise FileNotFoundError(f"Could not find data folder at: {data_root}")

image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".webp", ".heic", ".heif"}

stage_dirs = OrderedDict([
    ("raw", data_root / "1_raw"),
    ("standardized", data_root / "2_standarized"),
    ("train", data_root / "3_train"),
    ("test", data_root / "3_test"),
    ("processed_train", data_root / "4_processed_train"),
    ("processed_test", data_root / "4_processed_test"),
    ("roi_train", data_root / "5_roi_train"),
    ("roi_test", data_root / "5_roi_test"),
    ("enhanced_roi_train", data_root / "6_enhanced_roi_train"),
    ("enhanced_roi_test", data_root / "6_enhanced_roi_test"),
])


def count_stage_images(folder: Path, person_name: str) -> int:
    if not folder.exists():
        return 0

    person_dir = folder / person_name
    if person_dir.exists() and person_dir.is_dir():
        return sum(
            1
            for file_path in person_dir.iterdir()
            if file_path.is_file() and file_path.suffix.lower() in image_exts
        )

    return 0


def get_people() -> list[str]:
    raw_folder = stage_dirs["raw"]
    if raw_folder.exists():
        return sorted([p.name for p in raw_folder.iterdir() if p.is_dir()])

    for folder in stage_dirs.values():
        if folder.exists():
            return sorted([p.name for p in folder.iterdir() if p.is_dir()])

    return []


people = get_people()
if not people:
    print("No person folders found in the dataset.")
else:
    columns = ["Person", "Raw", "Std", "Train", "Test", "ROI Train", "ROI Test", "Enh Train", "Enh Test", "Retention %"]
    rows = []

    for person_name in people:
        raw_count = count_stage_images(stage_dirs["raw"], person_name)
        std_count = count_stage_images(stage_dirs["standardized"], person_name)
        train_count = count_stage_images(stage_dirs["train"], person_name)
        test_count = count_stage_images(stage_dirs["test"], person_name)
        roi_train_count = count_stage_images(stage_dirs["roi_train"], person_name)
        roi_test_count = count_stage_images(stage_dirs["roi_test"], person_name)
        enh_train_count = count_stage_images(stage_dirs["enhanced_roi_train"], person_name)
        enh_test_count = count_stage_images(stage_dirs["enhanced_roi_test"], person_name)

        final_total = enh_train_count + enh_test_count
        retention = (final_total / raw_count * 100.0) if raw_count else 0.0

        rows.append([
            person_name,
            raw_count,
            std_count,
            train_count,
            test_count,
            roi_train_count,
            roi_test_count,
            enh_train_count,
            enh_test_count,
            f"{retention:.1f}%",
        ])

    widths = [len(col) for col in columns]
    for row in rows:
        for index, value in enumerate(row):
            widths[index] = max(widths[index], len(str(value)))

    def format_row(values):
        return " | ".join(str(value).ljust(widths[index]) for index, value in enumerate(values))

    print(format_row(columns))
    print("-|-".join("-" * width for width in widths))
    for row in rows:
        print(format_row(row))

    print("=" * 110)
    total_raw = sum(int(row[1]) for row in rows)
    total_final = sum(int(row[7]) + int(row[8]) for row in rows)
    total_retention = (total_final / total_raw * 100.0) if total_raw else 0.0
    print(f"Total raw images      : {total_raw}")
    print(f"Total final ROI images: {total_final}")
    print(f"Overall retention     : {total_retention:.1f}%")
    print("Pipeline analysis complete.")

Person      | Raw | Std | Train | Test | ROI Train | ROI Test | Enh Train | Enh Test | Retention %
------------|-----|-----|-------|------|-----------|----------|-----------|----------|------------
benjamin    | 30  | 30  | 24    | 6    | 13        | 4        | 13        | 4        | 56.7%      
chern_tak   | 28  | 28  | 22    | 6    | 22        | 6        | 22        | 6        | 100.0%     
chillien    | 9   | 9   | 7     | 2    | 6         | 2        | 6         | 2        | 88.9%      
daniel      | 30  | 30  | 24    | 6    | 15        | 4        | 15        | 4        | 63.3%      
dylan       | 21  | 21  | 16    | 5    | 12        | 1        | 12        | 1        | 61.9%      
han_soon    | 20  | 20  | 16    | 4    | 15        | 4        | 15        | 4        | 95.0%      
harry       | 10  | 10  | 8     | 2    | 7         | 2        | 7         | 2        | 90.0%      
isaac       | 10  | 10  | 8     | 2    | 6         | 2        | 6         | 2        | 80.0%      
jing_ang  